# 🎙️ Soft Voting Baseline — K = 1 to 5

This notebook is part of the article **“DCMC-Fusion: Data-Centric and Model-Centric Ensemble Learning for Audio Deepfake Detection”**, by **Dora Ballesteros and Carlos Guerra**.

Its purpose is to evaluate **Soft Voting** as a non-trainable probability-fusion baseline for all architecture combinations and ensemble sizes **K=1,...,5** (31 configurations). Each prediction is based on the arithmetic mean of the selected **4K** detector scores, with **0 = natural** and **1 = synthetic**. The decision threshold is **50** on the 0–100 scale.

### 📥 Input Files

1. **`inference_probabilities.csv`**: probabilities from 20 previously trained detectors, generated independently of DCMC-Fusion.
2. **`experiment_split.json`**: saved train/test indices and dataset identity information from the hyperparameter-search workflow. **Only the held-out test partition is evaluated.**

### 📤 Output Files

1. **`soft_voting_global_metrics.csv`**: Accuracy, macro Precision, macro Recall and macro F1 for all 31 architecture combinations.
2. **`soft_voting_class_metrics.csv`**: Precision, Recall, F1 and Support for each class in every combination.

No model fitting or hyperparameter tuning is performed.

In [ ]:
from itertools import combinations
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

CSV_PATH = Path('inference_probabilities.csv')
SPLIT_PATH = Path('experiment_split.json')
OUTPUT_DIR = Path('.')
THRESHOLD = 50.0

ARCHITECTURES = {
    'resnet50': ['resnet50_mel', 'resnet50_log', 'resnet50_dwt', 'resnet50_cqt'],
    'densenet121': ['densenet121_mel', 'densenet121_log', 'densenet121_dwt', 'densenet121_cqt'],
    'inception_v3': ['inception_v3_mel', 'inception_v3_log', 'inception_v3_dwt', 'inception_v3_cqt'],
    'resnet34': ['resnet34_mel', 'resnet34_log', 'resnet34_dwt', 'resnet34_cqt'],
    'vgg16': ['vgg16_mel', 'vgg16_log', 'vgg16_dwt', 'vgg16_cqt'],
}
ARCH_ORDER = list(ARCHITECTURES)
FEATURE_COLUMNS = [col for cols in ARCHITECTURES.values() for col in cols]


## 1. Load and validate input data

In [ ]:
df = pd.read_csv(CSV_PATH)
required_columns = ['audio', 'label'] + FEATURE_COLUMNS
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f'Missing input columns: {missing}')

probabilities = df[FEATURE_COLUMNS].to_numpy(dtype=float)
if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 100)):
    raise ValueError('All detector scores must be finite and within [0, 100].')

labels = df['label'].to_numpy()
if set(np.unique(labels)) != {0, 1}:
    raise ValueError('Labels must contain both 0 (natural) and 1 (synthetic).')
labels = labels.astype(int)

with SPLIT_PATH.open(encoding='utf-8') as f:
    split = json.load(f)

if df['audio'].astype(str).tolist() != split['audio_order']:
    raise ValueError('Audio sample order differs from experiment_split.json.')
if labels.tolist() != split['label_order']:
    raise ValueError('Ground-truth labels differ from experiment_split.json.')
if FEATURE_COLUMNS != split['feature_columns']:
    raise ValueError('Feature columns differ from experiment_split.json.')

train_indices = np.asarray(split['train_indices'], dtype=int)
test_indices = np.asarray(split['test_indices'], dtype=int)
if (len(test_indices) == 0 or np.any(test_indices < 0) or
        np.any(test_indices >= len(df)) or
        len(np.unique(test_indices)) != len(test_indices)):
    raise ValueError('Invalid test indices.')
if np.intersect1d(train_indices, test_indices).size:
    raise ValueError('Training and test indices overlap.')

y_test = labels[test_indices]
print(f'All samples: {len(df)} | Held-out test samples: {len(test_indices)}')
print('Test class counts:', dict(zip(*np.unique(y_test, return_counts=True))))


All samples: 2484 | Held-out test samples: 746
Test class counts: {np.int64(0): np.int64(373), np.int64(1): np.int64(373)}


## 2. Evaluate Soft Voting for every K and architecture combination

For each sample, the mean of the selected 4K probabilities is compared with the threshold of 50. **Ties at 50 are classified as synthetic (class 1).**

In [ ]:
def soft_voting_predict(scores, threshold=THRESHOLD):
    """Mean detector score >= threshold indicates synthetic audio."""
    return (np.mean(scores, axis=1) >= threshold).astype(int)


global_rows = []
class_rows = []

for K in range(1, 6):
    for selected in combinations(ARCH_ORDER, K):
        architecture_name = ' + '.join(selected)
        columns = [col for arch in selected for col in ARCHITECTURES[arch]]
        X_test = df.iloc[test_indices][columns].to_numpy(dtype=float)
        predictions = soft_voting_predict(X_test)

        report = classification_report(
            y_test, predictions, labels=[0, 1],
            target_names=['natural', 'synthetic'],
            output_dict=True, zero_division=0
        )
        base = {'K': K, 'architectures': architecture_name, 'classifier': 'Soft-Voting'}
        macro = report['macro avg']
        global_rows.append({
            **base,
            'accuracy': accuracy_score(y_test, predictions),
            'precision_macro': macro['precision'],
            'recall_macro': macro['recall'],
            'f1_macro': macro['f1-score'],
        })
        for class_name in ['natural', 'synthetic']:
            metrics = report[class_name]
            class_rows.append({
                **base, 'class': class_name,
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1': metrics['f1-score'],
                'support': int(metrics['support']),
            })

global_metrics = pd.DataFrame(global_rows)
class_metrics = pd.DataFrame(class_rows)
assert len(global_metrics) == 31
assert len(class_metrics) == 62
print('Configurations per K:')
display(global_metrics.groupby('K').size().rename('configurations').to_frame())
print('Best Soft Voting configuration per K:')
display(global_metrics.loc[global_metrics.groupby('K')['f1_macro'].idxmax()].reset_index(drop=True))


Configurations per K:


,configurations
K,
1,5
2,10
3,10
4,5
5,1


Best Soft Voting configuration per K:


,K,architectures,classifier,accuracy,precision_macro,recall_macro,f1_macro
0,1,vgg16,Soft-Voting,0.596515,0.622967,0.596515,0.573582
1,2,inception_v3 + vgg16,Soft-Voting,0.583110,0.614367,0.583110,0.552536
2,3,resnet50 + inception_v3 + vgg16,Soft-Voting,0.581769,0.615703,0.581769,0.548678
3,4,resnet50 + densenet121 + inception_v3 + vgg16,Soft-Voting,0.567024,0.605362,0.567024,0.523696
4,5,resnet50 + densenet121 + inception_v3 + resnet...,Soft-Voting,0.554960,0.610128,0.554960,0.491245


## 3. Review distributions and export results

In [ ]:
display(global_metrics.groupby('K')['f1_macro'].agg(['count', 'mean', 'std', 'min', 'max']).fillna(0))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
global_path = OUTPUT_DIR / 'soft_voting_global_metrics.csv'
class_path = OUTPUT_DIR / 'soft_voting_class_metrics.csv'
global_metrics.to_csv(global_path, index=False)
class_metrics.to_csv(class_path, index=False)
print(f'Global metrics saved to: {global_path.resolve()}')
print(f'Class metrics saved to: {class_path.resolve()}')


,count,mean,std,min,max
K,,,,,
1,5,0.504478,0.070550,0.396560,0.573582
2,10,0.491647,0.051536,0.401343,0.552536
3,10,0.489736,0.037586,0.439746,0.548678
4,5,0.484755,0.023428,0.463851,0.523696
5,1,0.491245,0.000000,0.491245,0.491245


Global metrics saved to: /content/soft_voting_global_metrics.csv
Class metrics saved to: /content/soft_voting_class_metrics.csv
